In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.express as px
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_colwidth', None)

sys.path.insert(0, str(next((p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / 'utils').exists()), Path().resolve())))
import utils.imports as imports
import utils.features as features
import steps.isolation_forest as isolation_forest

In [2]:
config_path = "config/config.yml"

In [3]:
isolation_forest_preprocess = isolation_forest.IsolationForestPreprocess(config_path=config_path)
isolation_forest_data_df, isolation_features_list = isolation_forest_preprocess._execute()

data/bank_transactions_data.csv
C:\repos\research\anomaly_bank


In [4]:
isolation_forest_obj = isolation_forest.IsolationForestModel(config_path=config_path,
                                                              enriched_df=isolation_forest_data_df,
                                                              feat_columns=isolation_features_list)
best_features, anomaly_aggs_df, isolation_model, isolation_df, data_arr = isolation_forest_obj._execute()

100%|===================| 2508/2512 [00:11<00:00]       

anomaly_score_median                                                                                                                                                                                                                                                                                                                                                                                                                                                                         0.016845
anomaly_score_mean                                                                                                                                                                                                                                                                                                                                                                                                                                                                           0.015269
anomaly_score_stdev         

In [6]:
px.line(anomaly_aggs_df, x='feat_columns_count', y=['anomaly_score_median', 'anomaly_score_stdev', 'anomaly_score_mean', 'anomaly_final_score']).show()

In [7]:
# isolation_forest_data_df[['Anomaly', 'AnomalyScore']].head()
px.line(isolation_df.sort_values(by='Anomaly', ascending=True).reset_index(drop=True).reset_index(), 
        x='index', 
        y='AnomalyScore',
        color='Anomaly')

In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import shap
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import roc_auc_score  # Optional if labels exist

# -------------------------
# 1. Prepare Data
# -------------------------
features = [
    'TransactionAmount', 'TransactionDuration',
    'LoginAttempts', 'AccountBalance', 'CustomerAge'
]

X = df[features].fillna(0)

# -------------------------
# 2. Train Isolation Forest
# -------------------------
model = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
model.fit(X)

# Predict
df['Anomaly'] = model.predict(X)            # -1 = anomaly, 1 = normal
df['AnomalyScore'] = -model.decision_function(X)  # Higher = more anomalous

# -------------------------
# 3. SHAP Feature Importance
# -------------------------
explainer = shap.Explainer(model, X)
shap_values = explainer(X)

# Mean absolute SHAP values for global feature importance
importance_df = pd.DataFrame({
    'Feature': features,
    'Mean |SHAP value|': np.abs(shap_values.values).mean(axis=0)
}).sort_values(by='Mean |SHAP value|', ascending=True)

# Plotly bar chart
fig = px.bar(importance_df, x='Mean |SHAP value|', y='Feature', orientation='h',
             title='Global Feature Importance (SHAP)', height=400)
fig.show()

# -------------------------
# 4. Anomaly Score Distribution
# -------------------------
fig_score_dist = px.histogram(df, x='AnomalyScore', nbins=50,
                              color=df['Anomaly'].map({-1: 'Anomaly', 1: 'Normal'}),
                              title='Anomaly Score Distribution',
                              labels={'color': 'Prediction'})
fig_score_dist.show()

# -------------------------
# 5. Visualize Anomalies in 2D (Plotly scatter)
# -------------------------
# Use two most important features
top_features = importance_df['Feature'].values[-2:]
df['AnomalyScore'] = df['AnomalyScore'].fillna(0).clip(lower=0)

fig_scatter = px.scatter(df, x=top_features[0], y=top_features[1],
                         color=df['Anomaly'].map({-1: 'Anomaly', 1: 'Normal'}),
                         size='AnomalyScore',
                         title=f"Anomaly Visualization: {top_features[0]} vs {top_features[1]}",
                         hover_data=['TransactionID'] if 'TransactionID' in df.columns else None)
fig_scatter.show()

# -------------------------
# 6. Explore Top N Anomalies
# -------------------------
top_anomalies = df[df['Anomaly'] == -1].sort_values(by='AnomalyScore', ascending=False).head(5)
print("Top anomalies:")
print(top_anomalies[features + ['AnomalyScore']])

# -------------------------
# 7. (Optional) Evaluation with Metrics — Only if you have labels
# -------------------------
if 'IsFraud' in df.columns:
    from sklearn.metrics import classification_report, confusion_matrix

    y_true = df['IsFraud']
    y_pred = (df['Anomaly'] == -1).astype(int)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    # ROC AUC using anomaly scores
    roc_auc = roc_auc_score(y_true, df['AnomalyScore'])
    print(f"\nROC AUC Score: {roc_auc:.4f}")


 97%|=================== | 2427/2512 [00:15<00:00]       

Top anomalies:
      TransactionAmount  TransactionDuration  LoginAttempts  AccountBalance  \
394                6.30                  283              5         7697.68   
1557             262.43                  274              5        12841.01   
898             1531.31                   62              4          859.86   
1213            1192.20                  103              5         7816.41   
274             1176.28                  174              5          323.69   

      CustomerAge  AnomalyScore  
394            80      0.061537  
1557           37      0.042137  
898            18      0.040975  
1213           60      0.038609  
274            54      0.030659  
